In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi


#Lamb_shift = -0.01739466142645618 #Lamb shift in GHz
ω_b_0 = 2*ω_a #+ Lamb_shift*2*pi #7.7*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5

r=sqrt(Z_b/Z_a)

##Drive


ω_d=ω_b_0#-Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.3#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
terms_kept = [ (0, 0, 2, 3),
 (1, 1, 1, 2),
 (1, 2, 1, 1),
 (0, 0, 0, 3),
 (0, 0, 1, 2),
 (0, 1, 1, 1),
 (0, 2, 1, 0),
 (1, 1, 0, 1),
 (1, 2, 0, 0)]


Hamiltonian_term = [(i,j,k,l) for (i,j,k,l) in terms_kept]
for (i,j,k,l) in terms_kept
    push!(Hamiltonian_term, (j,i,l,k))
end
diff_a_coeff = Dict( 
    (i,j-1,k,l) => i+j+k+l == 3 ? 
        -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * factorial(i+j+k+l) / prod(factorial.((i,j,k,l))) * j : 
        1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * factorial(i+j+k+l) / prod(factorial.((i,j,k,l))) * j 
    for (i,j,k,l) in Hamiltonian_term if j != 0)
diff_b_coeff = Dict( 
    (i,j,k,l-1) => i+j+k+l == 3 ? 
        -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * factorial(i+j+k+l) / prod(factorial.((i,j,k,l))) * l * r : 
        1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * factorial(i+j+k+l) / prod(factorial.((i,j,k,l))) * l * r 
    for (i,j,k,l) in Hamiltonian_term if l != 0);

In [ ]:
factorial(3) / prod(factorial.((0,1,1,1)))

In [ ]:
#ATS + Mean_Field + ladder_operator
function excitation_to_number(excitation::NTuple{4,Int}, a::ComplexF32, b::ComplexF64)
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation
    return a^pow_a*conj(a)^pow_a_dag*b^pow_b*conj(b)^pow_b_dag
end
function test_equation_kept!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

function test_equation_minimum!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = -1/2*sqrt(Z_a^3/2)/ϕ_quantum^3*(r^2*b*conj(b)+a*(a/2+conj(a)+r*b+r*conj(b))+conj(a)*r*b)+1/8*sqrt(Z_a^5/2)/ϕ_quantum^5*(a*r^2*b*conj(b)*(a+2*conj(a)+r*b+r*conj(b)))

    #diff_b= sum(
    #excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    #for excitation in keys(diff_b_coeff)
    #)
    diff_b= -r/4*sqrt(Z_a^3/2)/ϕ_quantum^3*(2*r^2*b*conj(b)+2*a*conj(a)+2*(a+conj(a))*r*b+r^2*b^2+r^2*conj(b)^2+a^2)+r/48*sqrt(Z_a^5/2)/ϕ_quantum^5*(6*a*conj(a)*r*b*(a+conj(a)+r*b+2*r*conj(b))+3*r^4*b^2*conj(b)^2+2*r^4*b^3*conj(b))
        
    du[1] = -1im*(ω_a*a - sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

function test_equation_minimum_bis!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)
    φ_a = sqrt(Z_a/(2*ϕ_quantum^2))
    φ_b = sqrt(Z_b/(2*ϕ_quantum^2))

    sin_pump = E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)*φ_a^3

    diff_a =(a+r*b)*(a+2*conj(a)+r*b+2*r*conj(b))-r^2*b^2-φ_b^2*a*b*conj(b)*(a+2*conj(a)+r*b+r*conj(b))

    diff_b= (a+r*b)*(a+2*conj(a)+r*b+2*r*conj(b))+r^2*conj(b)^2-2*a*r*conj(b)-φ_b^2*b/r*(conj(a)*a*(a+conj(a)+r*b+2*r*conj(b))+r^3*b*conj(b)*(conj(b)/2+b/3))
        
    du[1] = -1im*(ω_a*a + sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + r*sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation_minimum_bis!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:
plot(real.(a_rot), imag.(a_rot))

# Reduction analysis

In [ ]:
excitation_sharing_list_full = Vector{NTuple{4,Int}}()
n = 5
for i in 0:n
    for j in 0:(n - i)
        for k in 0:(n - i - j)
            push!(excitation_sharing_list_full, (i, j, k, n - i - j - k))
        end
    end
end
multinomial_dict = Dict(
    excitation => factorial(n) / prod(factorial.(excitation))
    for excitation in excitation_sharing_list_full
);

In [ ]:
excitation_sharing_list = Vector{NTuple{4,Int}}()
n = 5
for i in 0:n
    for j in 0:(n - i)
        for k in 0:(n - i - j)
            if ! ((j ,i,  n - i - j - k, k) in excitation_sharing_list)
                push!(excitation_sharing_list, (i, j, k, n - i - j - k))
            end
        end
    end
end

In [ ]:
function excitation_to_monomial(excitation)
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation
    return latexstring("a^", "{$pow_a}","(a^\\dagger)", "{$pow_a_dag}","b^", "{$pow_b}","(b^\\dagger)", "{$pow_b_dag}")
end
function excitation_to_monomial(excitation) 
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation
    string_a, string_a_dag, string_b, string_b_dag = "", "", "", "" 
    if pow_a != 0 
        string_a = latexstring("a^", "{$pow_a}") 
    end 
    if pow_a_dag != 0 
        string_a_dag = latexstring("(a^\\dagger)^", "{$pow_a_dag}") 
    end 
    if pow_b != 0
        string_b = latexstring("b^", "{$pow_b}") 
    end 
    if pow_b_dag != 0
        string_b_dag = latexstring("(b^\\dagger)^", "{$pow_b_dag}") 
    end 
    return string_a*string_a_dag*string_b*string_b_dag end

In [ ]:
function get_max_monomial_list(a::Vector{ComplexF32},b::Vector{ComplexF32})
    max_monomial_list = Vector{Float64}()
    for excitation_sharing in excitation_sharing_list
        pow_a, pow_a_dag, pow_b, pow_b_dag = excitation_sharing
        coef = multinomial_dict[excitation_sharing]
        expr = @. a^pow_a * conj(a)^pow_a_dag * (r*b)^pow_b * (r*conj(b))^pow_b_dag
        max_monomial = maximum(abs.(expr))
        push!(max_monomial_list, coef*max_monomial)
    end
    return(max_monomial_list)
end

ϵ_0 = 0.5pi#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
r=sqrt(Z_b/Z_a)
t_max=20/κ_b
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a_0 = rand(Float32)/10 + 1im* rand(Float32)/10
b_0 = rand(Float32)/10 + 1im* rand(Float32)/10

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
a, b, _ = simulator(t_max, system, drive, environment)
max_monomial_list = get_max_monomial_list(a,b)
for i in 1:10
    a_0 = rand(Float32)/10 + 1im* rand(Float32)/10
    b_0 = rand(Float32)/10 + 1im* rand(Float32)/10

    system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
    a, b, _ = simulator(t_max, system, drive, environment)
    max_monomial_list_i = get_max_monomial_list(a,b)
    max_monomial_list += max_monomial_list_i
end
max_monomial_list /= 11;

In [ ]:
pairs = collect(zip(.-max_monomial_list, excitation_sharing_list))
pairs_sorted = sort(pairs; by = first)

# Recover sorted monomials
excitation_sharing_list_sorted = last.(pairs_sorted)
max_sorted = .-first.(pairs_sorted);

monomial_list = excitation_to_monomial.(excitation_sharing_list_sorted);

In [ ]:
plot(
    monomial_list,
    max_sorted,
    xrotation = 90,
    yaxis=:log,
    bottommargin = 10mm,
    label=""
)

# Changing the ODE

In [ ]:
terms_removed=vcat(excitation_sharing_list_sorted[end-17:end], excitation_sharing_list_sorted[end-23:end-19], excitation_sharing_list_sorted[end-26:end-26])
Hamiltonian_term = [(i,j,k,l) for (i,j,k,l) in terms_removed]
for (i,j,k,l) in terms_removed
    push!(Hamiltonian_term, (j,i,l,k))
end
diff_a_coeff = Dict( (i,j-1,k,l) => 1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * j for (i,j,k,l) in Hamiltonian_term if j != 0)
diff_b_coeff = Dict( (i,j,k,l-1) => 1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * l * r for (i,j,k,l) in Hamiltonian_term if l != 0);

In [ ]:
function excitation_to_number(excitation::NTuple{4,Int}, a::ComplexF32, b::ComplexF64)
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation
    return a^pow_a*conj(a)^pow_a_dag*b^pow_b*conj(b)^pow_b_dag
end

#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    coupling = (-1/2*((a+conj(a)) + r*(b+conj(b)))^2*Z_a/2/ϕ_quantum^2+1/24*(((a+conj(a)) + r*(b+conj(b)))^4)*Z_a^2/4/ϕ_quantum^4) / ϕ_quantum / sqrt(2)
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling + sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling + sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation3!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    coupling = -1/2*((a+conj(a)) + r*(b+conj(b)))^2*Z_a/2/ϕ_quantum^2 / ϕ_quantum / sqrt(2)
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
ϵ_0 = 0.8#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
t_max=20/κ_b
re_a_0, im_a_0 = 1, 1
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:

plot(real.(a_rot), imag.(a_rot))

In [ ]:
ϵ_0 = 0.3#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation3!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:
plot(real.(a_rot), imag.(a_rot))

In [ ]:
terms_kept = setdiff(excitation_sharing_list_sorted, terms_removed)

In [ ]:
Hamiltonian_term = [(i,j,k,l) for (i,j,k,l) in terms_kept]
for (i,j,k,l) in terms_kept
    push!(Hamiltonian_term, (j,i,l,k))
end
diff_a_coeff = Dict( (i,j-1,k,l) => 1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * j for (i,j,k,l) in Hamiltonian_term if j != 0)
diff_b_coeff = Dict( (i,j,k,l-1) => 1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * l * r for (i,j,k,l) in Hamiltonian_term if l != 0);

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation_kept!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    coupling = -1/2*((a+conj(a)) + r*(b+conj(b)))^2*Z_a/2/ϕ_quantum^2 / ϕ_quantum / sqrt(2)
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling - sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling - sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
ϵ_0 = 0.8#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation_kept!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:
plot(real.(a_rot), imag.(a_rot))

In [ ]:
terms_kept

# Changing the ODE

In [ ]:
excitation_sharing_list_full = Vector{NTuple{4,Int}}()
n = 3
for i in 0:n
    for j in 0:(n - i)
        for k in 0:(n - i - j)
            push!(excitation_sharing_list_full, (i, j, k, n - i - j - k))
        end
    end
end
multinomial_dict = Dict(
    excitation => factorial(n) / prod(factorial.(excitation))
    for excitation in excitation_sharing_list_full
);

In [ ]:
excitation_sharing_list = Vector{NTuple{4,Int}}()
n = 3
for i in 0:n
    for j in 0:(n - i)
        for k in 0:(n - i - j)
            if ! ((j ,i,  n - i - j - k, k) in excitation_sharing_list)
                push!(excitation_sharing_list, (i, j, k, n - i - j - k))
            end
        end
    end
end

In [ ]:
ϵ_0 = 0.5pi#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
r=sqrt(Z_b/Z_a)
t_max=20/κ_b
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a_0 = rand(Float32)/10 + 1im* rand(Float32)/10
b_0 = rand(Float32)/10 + 1im* rand(Float32)/10

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
a, b, _ = simulator(t_max, system, drive, environment)
max_monomial_list = get_max_monomial_list(a,b)
for i in 1:10
    a_0 = rand(Float32)/10 + 1im* rand(Float32)/10
    b_0 = rand(Float32)/10 + 1im* rand(Float32)/10

    system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
    a, b, _ = simulator(t_max, system, drive, environment)
    max_monomial_list_i = get_max_monomial_list(a,b)
    max_monomial_list += max_monomial_list_i
end
max_monomial_list /= 11;

In [ ]:
pairs = collect(zip(.-max_monomial_list, excitation_sharing_list))
pairs_sorted = sort(pairs; by = first)

# Recover sorted monomials
excitation_sharing_list_sorted = last.(pairs_sorted)
max_sorted = .-first.(pairs_sorted);

monomial_list = excitation_to_monomial.(excitation_sharing_list_sorted);

In [ ]:
plot(
    monomial_list,
    max_sorted,
    xrotation = 90,
    yaxis=:log,
    bottommargin = 10mm,
    label=""
)

In [ ]:
terms_removed=vcat(excitation_sharing_list_sorted[end:end],excitation_sharing_list_sorted[end-8:end-2])
Hamiltonian_term = [(i,j,k,l) for (i,j,k,l) in terms_removed]
for (i,j,k,l) in terms_removed
    push!(Hamiltonian_term, (j,i,l,k))
end
diff_a_coeff = Dict( (i,j-1,k,l) => -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * multinomial_dict[(i,j,k,l)] * j for (i,j,k,l) in Hamiltonian_term if j != 0)
diff_b_coeff = Dict( (i,j,k,l-1) => -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * multinomial_dict[(i,j,k,l)] * l * r for (i,j,k,l) in Hamiltonian_term if l != 0)

In [ ]:
function excitation_to_number(excitation::NTuple{4,Int}, a::ComplexF32, b::ComplexF64)
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation
    return a^pow_a*conj(a)^pow_a_dag*b^pow_b*conj(b)^pow_b_dag
end

#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    coupling = -1/2*((a+conj(a)) + r*(b+conj(b)))^2*Z_a/2/ϕ_quantum^2 / ϕ_quantum / sqrt(2)
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling + sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling + sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
ϵ_0 = 0.3#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:

plot(real.(a_rot), imag.(a_rot))

In [ ]:
terms_kept = setdiff(excitation_sharing_list_sorted, terms_removed)

In [ ]:

Hamiltonian_term = [(i,j,k,l) for (i,j,k,l) in terms_kept]
for (i,j,k,l) in terms_kept
    push!(Hamiltonian_term, (j,i,l,k))
end
diff_a_coeff = Dict( (i,j-1,k,l) => 1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * j for (i,j,k,l) in Hamiltonian_term if j != 0)
diff_b_coeff = Dict( (i,j,k,l-1) => 1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * l * r for (i,j,k,l) in Hamiltonian_term if l != 0);

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation_kept!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    coupling = -1/2*((a+conj(a)) + r*(b+conj(b)))^2*Z_a/2/ϕ_quantum^2 / ϕ_quantum / sqrt(2)
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling - sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling - sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
ϵ_0 = 0.8#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation_kept!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:
plot(real.(a_rot), imag.(a_rot))

# Finding minimum set-up for chaos and bistability

In [ ]:
n = 3
for i in 0:n
    for j in 0:(n - i)
        for k in 0:(n - i - j)
            push!(excitation_sharing_list_full, (i, j, k, n - i - j - k))
            multinomial_dict[(i, j, k, n - i - j - k)] = factorial(n) / prod(factorial.((i, j, k, n - i - j - k)))
        end
    end
end

In [ ]:
n = 3
for i in 0:n
    for j in 0:(n - i)
        for k in 0:(n - i - j)
            if ! ((j ,i,  n - i - j - k, k) in excitation_sharing_list_sorted)
                push!(excitation_sharing_list_sorted, (i, j, k, n - i - j - k))
            end
        end
    end
end

In [ ]:
for e in excitation_sharing_list_sorted
    display(e)
end

In [ ]:
terms_removed=vcat(excitation_sharing_list_sorted[end-10-17:end-10],
    excitation_sharing_list_sorted[end-10-23:end-10-19], 
    excitation_sharing_list_sorted[end-10-26:end-10-26],
    excitation_sharing_list_sorted[end-2:end-2],
    excitation_sharing_list_sorted[end-5:end-5],
    excitation_sharing_list_sorted[end-7:end-7])
Hamiltonian_term = [(i,j,k,l) for (i,j,k,l) in terms_removed]
for (i,j,k,l) in terms_removed
    push!(Hamiltonian_term, (j,i,l,k))
end
diff_a_coeff = Dict( 
    (i,j-1,k,l) => i+j+k+l == 3 ? 
        -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * multinomial_dict[(i,j,k,l)] * j : 
        1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * j 
    for (i,j,k,l) in Hamiltonian_term if j != 0)
diff_b_coeff = Dict( 
    (i,j,k,l-1) => i+j+k+l == 3 ? 
        -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * multinomial_dict[(i,j,k,l)] * l * r : 
        1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * l * r 
    for (i,j,k,l) in Hamiltonian_term if l != 0)

In [ ]:
function excitation_to_number(excitation::NTuple{4,Int}, a::ComplexF32, b::ComplexF64)
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation
    return a^pow_a*conj(a)^pow_a_dag*b^pow_b*conj(b)^pow_b_dag
end

#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    coupling = (-1/2*((a+conj(a)) + r*(b+conj(b)))^2*Z_a/2/ϕ_quantum^2+1/24*(((a+conj(a)) + r*(b+conj(b)))^4)*Z_a^2/4/ϕ_quantum^4) / ϕ_quantum / sqrt(2)
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling + sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling + sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
ϵ_0 = 0.8#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0#-2-1im

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:

plot(real.(a_rot), imag.(a_rot))

In [ ]:
terms_kept = setdiff(excitation_sharing_list_sorted, terms_removed)

In [ ]:

Hamiltonian_term = [(i,j,k,l) for (i,j,k,l) in terms_kept]
for (i,j,k,l) in terms_kept
    push!(Hamiltonian_term, (j,i,l,k))
end
diff_a_coeff = Dict( 
    (i,j-1,k,l) => i+j+k+l == 3 ? 
        -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * multinomial_dict[(i,j,k,l)] * j : 
        1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * j 
    for (i,j,k,l) in Hamiltonian_term if j != 0)
diff_b_coeff = Dict( 
    (i,j,k,l-1) => i+j+k+l == 3 ? 
        -1/6 * (sqrt(Z_a/2)/ϕ_quantum)^3 * multinomial_dict[(i,j,k,l)] * l * r : 
        1/120 * (sqrt(Z_a/2)/ϕ_quantum)^5 * multinomial_dict[(i,j,k,l)] * l * r 
    for (i,j,k,l) in Hamiltonian_term if l != 0);

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation_kept!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)

    coupling = 0.0
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)

    diff_a = sum(
    excitation_to_number(excitation, a, r*b) * diff_a_coeff[excitation]
    for excitation in keys(diff_a_coeff)
    )

    diff_b= sum(
    excitation_to_number(excitation, a, r*b) * diff_b_coeff[excitation]
    for excitation in keys(diff_b_coeff)
    )
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling - sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling - sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
ϵ_0 = 0.8#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
t_max=20/κ_b
re_a_0, im_a_0 = 2, 1
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation_kept!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:
plot(real.(a_rot), imag.(a_rot))